# 🛡️ News Article Classification — Fake vs Real

**Objective:** Classify news articles as Fake (0) or Real (1) using NLP + Machine Learning  
**Dataset:** [Kaggle — Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)  
**Tools:** Python · NLTK · scikit-learn · TF-IDF · Logistic Regression · Naive Bayes

---

## 📋 Table of Contents
1. Setup & Imports
2. Data Loading & Exploration
3. Text Preprocessing (NLTK)
4. TF-IDF Feature Extraction
5. Model Training
6. Evaluation & Metrics
7. Visualisations
8. Save Best Model

## 1. Setup & Imports

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Add project root to path so we can use the src/ modules
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from wordcloud import WordCloud
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_curve, auc
)

# Project modules
from src.data_preprocessing import (
    load_raw_data, preprocess_data, split_data
)
from src.feature_engineering import fit_transform_vectorizer, save_vectorizer
from src.model_training import train_all_models, save_models, save_best_model
from src.evaluation import (
    evaluate_all_models, select_best_model,
    plot_confusion_matrix, plot_roc_curve, plot_model_comparison
)
from src.utils import save_metrics
from src.config import METRICS_PATH

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('✅ All imports successful')

## 2. Data Loading & Exploration

In [ ]:
df = load_raw_data()
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# --- Class distribution ---
counts = df['label'].value_counts()
label_names = {0: 'FAKE', 1: 'REAL'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(
    [label_names[k] for k in counts.index], counts.values,
    color=['#ef4444', '#22c55e'], edgecolor='white', linewidth=1.5
)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 150, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts.values, labels=[label_names[k] for k in counts.index],
    autopct='%1.1f%%', colors=['#ef4444', '#22c55e'],
    startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Class Balance', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'\nFake: {counts[0]:,} | Real: {counts[1]:,} | Balance: {counts[0]/counts[1]:.2f}')

In [ ]:
# Article length distribution
df['word_count'] = df['content'].str.split().str.len()

fig, ax = plt.subplots(figsize=(10, 4))
for label, color in [(0, '#ef4444'), (1, '#22c55e')]:
    subset = df[df['label'] == label]['word_count']
    ax.hist(subset.clip(upper=2000), bins=60, alpha=0.6,
            color=color, label=label_names[label])
ax.set_xlabel('Word Count (clipped at 2000)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Article Length Distribution', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby('label')['word_count'].describe().round(1))

In [ ]:
# --- Word Clouds ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, color_map, title) in zip(axes, [
    (0, 'Reds',  '☠️  FAKE News — Top Words'),
    (1, 'Greens','✅ REAL News — Top Words'),
]):
    text = ' '.join(df[df['label'] == label]['content'].dropna().astype(str))
    wc = WordCloud(
        width=700, height=400, background_color='white',
        max_words=100, colormap=color_map
    ).generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold')

plt.suptitle('Word Clouds by Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Text Preprocessing (NLTK)

In [ ]:
# Show cleaning effect with one example
from src.data_preprocessing import clean_text

sample = df.iloc[0]['content']
print('─── Original (first 400 chars) ───')
print(sample[:400])
print('\n─── After Cleaning ───')
print(clean_text(sample)[:400])

In [ ]:
# Preprocess full dataset
df = preprocess_data(df)
print(df[['content', 'cleaned_content', 'label']].head(3))

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = split_data(df)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Test class balance: {y_test.value_counts().to_dict()}')

## 4. TF-IDF Feature Extraction

In [ ]:
vectorizer, X_train_tfidf, X_test_tfidf = fit_transform_vectorizer(X_train, X_test)
save_vectorizer(vectorizer)

# Show top terms by TF-IDF score
import numpy as np
feature_names = vectorizer.get_feature_names_out()
mean_tfidf = np.asarray(X_train_tfidf.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[-20:][::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(feature_names[top_idx][::-1], mean_tfidf[top_idx][::-1], color='#7c3aed')
ax.set_title('Top 20 Terms by Mean TF-IDF Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean TF-IDF')
plt.tight_layout()
plt.show()

## 5. Model Training

In [ ]:
models = train_all_models(X_train_tfidf, y_train)
save_models(models)
print('Models:', list(models.keys()))

## 6. Evaluation & Metrics

In [ ]:
all_metrics = evaluate_all_models(models, X_test_tfidf, y_test)

# Summary table
summary = pd.DataFrame([
    {
        'Model':         name,
        'Accuracy (%)':  m['accuracy'],
        'F1 Macro (%)':  m['f1_macro'],
        'F1 Weighted (%)': m['f1_weighted'],
        'Precision (%)': m['precision_macro'],
        'Recall (%)':    m['recall_macro'],
    }
    for name, m in all_metrics.items()
])
summary.set_index('Model', inplace=True)
summary.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}')

## 7. Visualisations

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['FAKE', 'REAL'],
        yticklabels=['FAKE', 'REAL'], ax=ax
    )
    acc = accuracy_score(y_test, y_pred) * 100
    ax.set_title(f'{name}\nAccuracy: {acc:.2f}%', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plot_roc_curve(models, X_test_tfidf, y_test)
plt.show()

In [ ]:
# Model comparison bar chart
plot_model_comparison(all_metrics)
plt.show()

In [ ]:
# Top LR features: FAKE vs REAL
lr_model = models['Logistic Regression']
feature_names = vectorizer.get_feature_names_out()
coef = lr_model.coef_[0]

N = 20
top_real_idx = coef.argsort()[-N:][::-1]
top_fake_idx = coef.argsort()[:N]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, idx, title, color in [
    (axes[0], top_real_idx, '✅ Top REAL indicators', '#22c55e'),
    (axes[1], top_fake_idx, '🚨 Top FAKE indicators', '#ef4444'),
]:
    words   = feature_names[idx]
    weights = coef[idx]
    ax.barh(words[::-1], weights[::-1], color=color, alpha=0.8)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('LR Coefficient')

plt.suptitle('Logistic Regression — Most Informative Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Save Best Model

In [ ]:
best_name, best_model = select_best_model(models, all_metrics)
save_best_model(best_model, best_name)

all_metrics['best_model'] = best_name
save_metrics(all_metrics, METRICS_PATH)

print(f'\n🎉 Best model: {best_name}')
print(f'   Accuracy      : {all_metrics[best_name]["accuracy"]:.4f}%')
print(f'   F1 (weighted) : {all_metrics[best_name]["f1_weighted"]:.4f}%')
print('\nAll model artifacts saved to models/')

---
## 🚀 Next Steps
1. Run the Streamlit app: `streamlit run app.py`
2. Paste any news article to get a real-time Fake/Real prediction with explanation
3. Try improving accuracy with: GridSearchCV, PassiveAggressive Classifier, or BERT embeddings